# Config B — Validation, Calibration, Window Size
### Three methodologically sound experiments

Config B reached macro-F1 0.543. Three things worth fixing or testing before chasing the
number further.

**Experiment 1 — Group-aware validation.**
Current training uses `validation_split=0.15`, which takes the last 15% of the training
array *before shuffling*. Because windows overlap by 96% (window 120, step 5) and arrays
are ordered subject-by-subject, that validation set is an uncontrolled slice whose subject
composition changes with every fold. Early stopping is therefore decided on something
arbitrary. The fix is to hold out whole subjects for validation.

*An honest confound:* holding out 2 subjects for validation means training on 12 instead
of 14. The comparison below measures the combined effect of proper validation and reduced
training data — these cannot be separated, and the result may well be lower than 0.543.
That would still be the more trustworthy number.

**Experiment 2 — Rank-threshold calibration.**
CORN decodes with a fixed 0.5 cut on each cumulative rank probability. That threshold is
arbitrary and is not the rule that maximises macro-F1 under class imbalance. Thresholds
are fitted on the held-out *validation subjects* and applied unchanged to the test
subject, so no test information is used.

**Experiment 3 — Window size.**
120 beats is roughly 1.5–2 minutes, which may blur stress transitions. Tests 60, 120 and
180 beats. Step stays at 5 throughout, so temporal sampling density is constant while
window overlap differs (92%, 96%, 97%) — noted as a caveat rather than controlled away.

**Not included, and why.** Loss-function and focal-gamma comparisons do not apply: config B
uses a CORN ordinal head with binary cross-entropy over rank tasks, not focal loss with
argmax. Label regeneration is excluded because the current labels have already been
validated and changing them would break comparability with all prior results.


## 1. Configuration

In [1]:
!pip install neurokit2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 11.6 MB/s eta 0:00:00


In [2]:
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import wilcoxon
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import f1_score, cohen_kappa_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
import neurokit2 as nk
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

DATA_PATH='/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE='/kaggle/working'; os.makedirs(SAVE,exist_ok=True)
SUBJECT_IDS=[2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES=['relaxed','mild','moderate','high']; NCLS=4
STEP=5; LATENT_DIM=128
N_VAL_SUBJECTS=2          # held out from the 14 training subjects
SEED=42
RESULTS=f'{SAVE}/configB_valcal.json'
print("TF",tf.__version__,"GPU",len(tf.config.list_physical_devices('GPU'))>0)

TF 2.20.0 GPU True


## 2. Preprocessing

In [3]:
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl",'rb') as f: d=pickle.load(f,encoding='latin1')
    return d['signal']['chest'], d['signal']['wrist']['TEMP'].flatten(), d['label'].flatten()
def extract_rr(ecg,fs=700):
    ecg=nk.ecg_clean(ecg.flatten(),sampling_rate=fs); _,i=nk.ecg_peaks(ecg,sampling_rate=fs); rp=i['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs),(rp[:-1]+rp[1:])/2.0/fs,rp
def clean_rr(rr,ts):
    rr=rr.copy().astype(float); rr[(rr<=300)|(rr>=2000)]=np.nan
    for i in range(1,len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]) and abs(rr[i]-rr[i-1])/rr[i-1]>0.20: rr[i]=np.nan
    m=np.isnan(rr)
    if m.any(): rr[m]=np.interp(np.where(m)[0],np.where(~m)[0],rr[~m])
    return rr,ts
def align_temp(wt,rp,fe=700,ft=4.0):
    tap=np.interp(rp/fe,np.arange(len(wt))/ft,wt); return (tap[:-1]+tap[1:])/2.0
def labels_to_rr(labels,rp):
    o=[]
    for i in range(len(rp)-1):
        seg=labels[rp[i]:rp[i+1]]; v=seg[seg>0]; o.append(0 if len(v)==0 else np.bincount(v).argmax())
    return np.array(o)
def cos_model(th,m,a,p): return m+a*np.cos((2*np.pi/24.0)*th+p)
def fit_cos(sig,ts,p0):
    th=(ts%86400)/3600.0
    try:
        popt,_=curve_fit(cos_model,th,sig,p0=p0,maxfev=10000); return cos_model(th,*popt)
    except RuntimeError: return np.full_like(sig,np.mean(sig))
def roll_stat(rr,fn):
    o=np.zeros(len(rr))
    for i in range(len(rr)): o[i]=fn(rr[max(0,i-10):i+10])
    return o
def roll_rmssd(rr): return roll_stat(rr,lambda w:(np.sqrt(np.mean(np.diff(w)**2)) if len(w)>1 else 0))
def roll_sdnn(rr):  return roll_stat(rr,lambda w:(np.std(w) if len(w)>1 else 0))

wesad={}
for sid in SUBJECT_IDS:
    try:
        chest,wt,labels=load_subject(sid); ecg=chest['ECG'].flatten()
        rr,ts,rp=extract_rr(ecg); temp=align_temp(wt,rp); rr,ts=clean_rr(rr,ts)
        rl=labels_to_rr(labels,rp); keep=rl>0
        rrk,tk,tsk,lk=rr[keep],temp[keep],ts[keep],rl[keep]
        new=np.zeros(len(lk),dtype=int); si=np.where(lk==2)[0]
        if len(si)>0:
            srr=rrk[si]; loc=[]
            for i in range(len(srr)):
                w=srr[max(0,i-15):i+15]; dd=np.diff(w); loc.append(np.sqrt(np.mean(dd**2)) if len(dd)>0 else 50)
            loc=np.array(loc); p33,p66=np.percentile(loc,33),np.percentile(loc,66)
            for i,idx in enumerate(si): new[idx]=(1 if loc[i]>=p66 else 2 if loc[i]>=p33 else 3)
        wesad[f'S{sid}']=dict(rr_ms=rrk,temp=tk,timestamps=tsk,labels=new,
                              exp_rr=fit_cos(rrk,tsk,[np.mean(rrk),50,-1.5]),
                              exp_temp=fit_cos(tk,tsk,[np.mean(tk),1,-1.5]))
    except Exception as e: print("FAIL",sid,e)
print(len(wesad),"subjects"); assert len(wesad)==15

15 subjects


### Window builder, parameterised by window size

In [4]:
EXP_ALL=['exp_rr_mean','exp_rr_std','exp_rr_slope','exp_tp_mean','exp_tp_std',
         'exp_tp_slope','sin_24h','cos_24h','sin_90m','cos_90m','cortisol']

def build_inputs(window, step=STEP):
    Xs,Xe,y,g=[],[],[],[]
    for sid,d in wesad.items():
        rr,temp=d['rr_ms'],d['temp']; labels,ts=d['labels'],d['timestamps']
        er,et=d['exp_rr'],d['exp_temp']
        mu_r,sd_r=np.mean(rr),np.std(rr)+1e-8; mu_t,sd_t=np.mean(temp),np.std(temp)+1e-8
        rn=(rr-mu_r)/sd_r; tn=(temp-mu_t)/sd_t
        rm,sd=roll_rmssd(rn),roll_sdnn(rn)
        hr=60000/(rr+1e-8); hrn=(hr-np.mean(hr))/(np.std(hr)+1e-8)
        ern=(er-mu_r)/sd_r; etn=(et-mu_t)/sd_t; idx=np.arange(window)
        for s in range(0,len(rr)-window,step):
            e=s+window; bi=min(s+window//2,len(ts)-1)
            Xs.append(np.stack([rn[s:e],rm[s:e],sd[s:e],hrn[s:e],tn[s:e]],axis=-1))
            wr,wt_=ern[s:e],etn[s:e]; tt=ts[bi]%86400; hour=tt/3600.0
            Xe.append(np.array([wr.mean(),wr.std(),np.polyfit(idx,wr,1)[0],
                                wt_.mean(),wt_.std(),np.polyfit(idx,wt_,1)[0],
                                np.sin(2*np.pi*tt/86400),np.cos(2*np.pi*tt/86400),
                                np.sin(2*np.pi*tt/5400),np.cos(2*np.pi*tt/5400),
                                0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2)],
                               dtype=np.float32))
            y.append(labels[s+window//2]); g.append(int(sid[1:]))
    Xs=np.array(Xs,dtype=np.float32); Xe=np.array(Xe,dtype=np.float32)
    alive=Xe.std(axis=0)>1e-3
    return Xs,Xe[:,alive],np.array(y,dtype=np.int32),np.array(g,dtype=np.int32),\
           [EXP_ALL[i] for i in np.where(alive)[0]]
print("builder ready")

builder ready


## 3. Config B architecture and CORN utilities

In [5]:
def build_model(window,n_exp,n_seq_ch=5,latent=LATENT_DIM,dropout=0.4):
    seq_in=tf.keras.Input((window,n_seq_ch),name='sequence')
    exp_in=tf.keras.Input((n_exp,),name='expected')
    x=layers.Conv1D(64,7,padding='same',activation='relu')(seq_in)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x)
    x=layers.Dropout(dropout)(x); x=layers.GlobalAveragePooling1D()(x)
    H_cur=layers.Dense(latent,activation='relu',name='H_current')(x)
    e=layers.Dense(64,activation='relu')(exp_in); e=layers.Dense(64,activation='relu')(e)
    H_exp=layers.Dense(latent,activation='relu',name='H_expected')(e)
    H_dev=layers.Subtract(name='H_deviation')([H_cur,H_exp])
    st=layers.Lambda(lambda t: tf.stack(t,axis=1),name='stack')([H_cur,H_exp,H_dev])
    at=layers.Attention()([st,st]); pooled=layers.Flatten()(at)
    h=layers.Dense(64,activation='relu')(pooled); h=layers.Dropout(dropout)(h)
    return Model([seq_in,exp_in],layers.Dense(NCLS-1,activation=None,name='corn')(h))

def corn_labels(y,K=NCLS):
    y=np.asarray(y).reshape(-1,1); return (y>np.arange(K-1).reshape(1,-1)).astype(np.float32)
def corn_loss(yb,lo): return tf.reduce_mean(tf.keras.losses.binary_crossentropy(yb,tf.sigmoid(lo)))

def corn_decode_thr(logits,thr=None):
    """Rank-monotonic decode. Counts LEADING passes, stopping at the first failure,
    so a non-monotonic threshold vector cannot produce an incoherent class."""
    thr=np.full(NCLS-1,0.5) if thr is None else np.asarray(thr)
    p=1.0/(1.0+np.exp(-np.asarray(logits)))
    passed=np.cumprod(p,axis=1)>thr[None,:]
    pad=np.concatenate([passed,np.zeros((len(passed),1),dtype=bool)],axis=1)
    return np.argmin(pad,axis=1).astype(int)

def macro_f1_fast(yt,yp,K=NCLS):
    f=[]
    for c in range(K):
        tp=np.sum((yp==c)&(yt==c)); fp=np.sum((yp==c)&(yt!=c)); fn=np.sum((yp!=c)&(yt==c))
        f.append(0.0 if tp==0 else 2*tp/(2*tp+fp+fn))
    return float(np.mean(f))

def ordinal_metrics(yt,yp):
    yt,yp=np.asarray(yt),np.asarray(yp); err=np.abs(yp-yt)
    return dict(f1=f1_score(yt,yp,average='macro',zero_division=0),
                kappa=cohen_kappa_score(yt,yp,weights='quadratic'),
                mae=float(err.mean()),
                ma_mae=float(np.mean([err[yt==c].mean() for c in range(NCLS) if np.any(yt==c)])),
                dist=float((err>=2).mean()))

# decode sanity
_t=np.log(np.array([[.1,.1,.1],[.9,.1,.1],[.9,.9,.1],[.9,.9,.9]])/(1-np.array([[.1,.1,.1],[.9,.1,.1],[.9,.9,.1],[.9,.9,.9]])))
print("decode round-trip:",corn_decode_thr(_t),"(expect [0 1 2 3])")

decode round-trip: [0 1 2 3] (expect [0 1 2 3])


## 4. Threshold calibration
Grid search over the three rank thresholds, maximising macro-F1 **on the validation
subjects only**. The test subject is never involved in fitting.

In [6]:
GRID=np.arange(0.20,0.81,0.05)

def fit_thresholds(val_logits,val_y):
    best,best_f1=np.full(NCLS-1,0.5),-1.0
    p=1.0/(1.0+np.exp(-val_logits)); pc=np.cumprod(p,axis=1)
    for t1 in GRID:
        m1=pc[:,0]>t1
        for t2 in GRID:
            m2=pc[:,1]>t2
            for t3 in GRID:
                passed=np.stack([m1,m2,pc[:,2]>t3],axis=1)
                pad=np.concatenate([passed,np.zeros((len(passed),1),dtype=bool)],axis=1)
                pred=np.argmin(pad,axis=1)
                f=macro_f1_fast(val_y,pred)
                if f>best_f1: best_f1,best=f,np.array([t1,t2,t3])
    return best,best_f1
print(f"grid: {len(GRID)}^3 = {len(GRID)**3} combinations per fold")

grid: 13^3 = 2197 combinations per fold


## 5. Fold construction and training
Two validation strategies. `old` reproduces the current behaviour; `group` holds out whole
subjects, which costs two subjects of training data — the confound noted at the top.

In [7]:
def make_folds(groups,strategy,n_val=N_VAL_SUBJECTS,seed=SEED):
    logo=LeaveOneGroupOut(); rng=np.random.RandomState(seed)
    for tr_all,te in logo.split(np.zeros(len(groups)),groups=groups):
        if strategy=='old':
            yield tr_all,None,te
        else:
            tr_subj=np.unique(groups[tr_all])
            val_subj=rng.choice(tr_subj,n_val,replace=False)
            vm=np.isin(groups,val_subj)
            yield tr_all[~vm[tr_all]], tr_all[vm[tr_all]], te

def train_fold(Xs,Xe,y,g,tr,va,te,window,seed=SEED,dropout=0.4):
    tf.keras.backend.clear_session(); np.random.seed(seed); tf.random.set_seed(seed)
    cw=dict(enumerate(compute_class_weight('balanced',classes=np.unique(y[tr]),y=y[tr])))
    sw=np.array([cw[c] for c in y[tr]],dtype=np.float32)
    sc=StandardScaler().fit(Xe[tr])
    m=build_model(window,Xe.shape[1],dropout=dropout)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=lambda a,b:corn_loss(a,b))
    cb=[callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True,verbose=0)]
    fit_kw=dict(sample_weight=sw,epochs=100,batch_size=32,callbacks=cb,verbose=0)
    if va is None:
        m.fit([Xs[tr],sc.transform(Xe[tr])],corn_labels(y[tr]),validation_split=0.15,**fit_kw)
        val_logits,val_y=None,None
    else:
        m.fit([Xs[tr],sc.transform(Xe[tr])],corn_labels(y[tr]),
              validation_data=([Xs[va],sc.transform(Xe[va])],corn_labels(y[va])),**fit_kw)
        val_logits=m.predict([Xs[va],sc.transform(Xe[va])],verbose=0); val_y=y[va]
    test_logits=m.predict([Xs[te],sc.transform(Xe[te])],verbose=0)
    return test_logits,val_logits,val_y

def run_loso(Xs,Xe,y,g,window,strategy,label,store,calibrate=True):
    if label in store: print(f"{label} cached"); return
    yt,yp,ypc,pf,pfc,thr_log=[],[],[],{},{},[]
    print(f"\n{label}",end='  ')
    for tr,va,te in make_folds(g,strategy):
        s=int(np.unique(g[te])[0]); print(f"S{s:02d}",end=' ',flush=True)
        tl,vl,vy=train_fold(Xs,Xe,y,g,tr,va,te,window)
        p05=corn_decode_thr(tl)
        yt.extend(y[te]); yp.extend(p05); pf[str(s)]=float(macro_f1_fast(y[te],p05))
        if calibrate and vl is not None:
            thr,vf=fit_thresholds(vl,vy); thr_log.append(thr.tolist())
            pc_=corn_decode_thr(tl,thr); ypc.extend(pc_); pfc[str(s)]=float(macro_f1_fast(y[te],pc_))
    rec=dict(window=window,strategy=strategy,metrics=ordinal_metrics(yt,yp),perfold=pf,
             y_true=[int(v) for v in yt],y_pred=[int(v) for v in yp])
    if ypc:
        rec['metrics_cal']=ordinal_metrics(yt,ypc); rec['perfold_cal']=pfc
        rec['thresholds']=thr_log; rec['y_pred_cal']=[int(v) for v in ypc]
    store[label]=rec; json.dump(store,open(RESULTS,'w'))
    msg=f"\n   F1={rec['metrics']['f1']:.3f}"
    if ypc: msg+=f"   calibrated F1={rec['metrics_cal']['f1']:.3f}"
    print(msg+"  [saved]")
print("runner ready")

runner ready


## 6. Experiments 1 and 2 — validation strategy and calibration
Both come from the same run: the group-aware pass produces validation predictions, which
are used to fit thresholds, which are then applied to the untouched test subject.

In [8]:
store=json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
Xs120,Xe120,y120,g120,names120=build_inputs(120)
print("window 120:",Xs120.shape,Xe120.shape,"| kept:",names120)

t0=time.time()
run_loso(Xs120,Xe120,y120,g120,120,'old',  'w120_old',   store,calibrate=False)
run_loso(Xs120,Xe120,y120,g120,120,'group','w120_group', store,calibrate=True)
print(f"\nelapsed {(time.time()-t0)/60:.0f} min")

window 120: (11846, 120, 5) (11846, 9) | kept: ['exp_rr_mean', 'exp_rr_std', 'exp_tp_mean', 'exp_tp_std', 'exp_tp_slope', 'sin_24h', 'cos_24h', 'sin_90m', 'cos_90m']

w120_old  S02 

I0000 00:00:1784884320.490821      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784884320.493474      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


S03 S04 S05 S06 S07 S08 S09 S10 S11 S13 S14 S15 S16 S17 
   F1=0.549  [saved]

w120_group  S02 S03 S04 S05 S06 S07 S08 S09 S10 S11 S13 S14 S15 S16 S17 
   F1=0.552   calibrated F1=0.552  [saved]

elapsed 38 min


In [9]:
a=store['w120_old']['metrics']; b=store['w120_group']['metrics']
c=store['w120_group'].get('metrics_cal')
print("="*70); print("EXPERIMENT 1 - VALIDATION STRATEGY"); print("="*70)
print(f"{'metric':<12}{'old split':>14}{'group-aware':>14}{'delta':>12}")
for k in ['f1','kappa','ma_mae','dist']:
    print(f"{k:<12}{a[k]:>14.3f}{b[k]:>14.3f}{b[k]-a[k]:>+12.3f}")
print("\nNote: group-aware also trains on 12 subjects instead of 14, so a drop here")
print("reflects BOTH the stricter validation and the reduced training data.")

if c:
    print("\n"+"="*70); print("EXPERIMENT 2 - THRESHOLD CALIBRATION"); print("="*70)
    print(f"{'metric':<12}{'0.5 default':>14}{'calibrated':>14}{'delta':>12}")
    for k in ['f1','kappa','ma_mae','dist']:
        print(f"{k:<12}{b[k]:>14.3f}{c[k]:>14.3f}{c[k]-b[k]:>+12.3f}")
    thr=np.array(store['w120_group']['thresholds'])
    print(f"\nfitted thresholds, mean across folds: {thr.mean(axis=0).round(3)}")
    print(f"                   sd  across folds: {thr.std(axis=0).round(3)}")
    print("High sd means the optimal threshold is unstable between folds, which")
    print("would make the calibration unreliable in deployment.")
    subs=sorted(store['w120_group']['perfold'])
    u=np.array([store['w120_group']['perfold_cal'][s] for s in subs])
    v=np.array([store['w120_group']['perfold'][s] for s in subs])
    try:
        _,p=wilcoxon(u,v); d=(u-v).mean()/((u-v).std(ddof=1)+1e-12)
        print(f"\nWilcoxon calibrated vs default: p={p:.4f}  d={d:+.2f}  dF1={(u-v).mean():+.4f}")
    except Exception as e: print("test skipped:",e)

EXPERIMENT 1 - VALIDATION STRATEGY
metric           old split   group-aware       delta
f1                   0.549         0.552      +0.003
kappa                0.830         0.828      -0.002
ma_mae               0.555         0.518      -0.036
dist                 0.033         0.032      -0.002

Note: group-aware also trains on 12 subjects instead of 14, so a drop here
reflects BOTH the stricter validation and the reduced training data.

EXPERIMENT 2 - THRESHOLD CALIBRATION
metric         0.5 default    calibrated       delta
f1                   0.552         0.552      -0.000
kappa                0.828         0.809      -0.018
ma_mae               0.518         0.569      +0.051
dist                 0.032         0.043      +0.011

fitted thresholds, mean across folds: [0.517 0.54  0.343]
                   sd  across folds: [0.217 0.128 0.109]
High sd means the optimal threshold is unstable between folds, which
would make the calibration unreliable in deployment.

Wilcoxon cali

## 7. Experiment 3 — window size
60, 120 and 180 beats, all with group-aware validation and calibration. Step stays at 5,
so overlap differs by window (92%, 96%, 97%) and the number of windows differs — pooled
metrics are therefore over different sample counts, while per-subject F1 remains
comparable.

In [10]:
for w in [60,180]:
    Xs,Xe,y,g,nm=build_inputs(w)
    print(f"\nwindow {w}: {Xs.shape[0]} windows, exp dim {Xe.shape[1]}")
    run_loso(Xs,Xe,y,g,w,'group',f'w{w}_group',store,calibrate=True)
    del Xs,Xe

rows=[]
for w in [60,120,180]:
    k=f'w{w}_group'
    if k not in store: continue
    r=store[k]
    rows.append(dict(window=w,n_windows=len(r['y_true']),
                     f1=r['metrics']['f1'],kappa=r['metrics']['kappa'],
                     ma_mae=r['metrics']['ma_mae'],
                     f1_cal=r.get('metrics_cal',{}).get('f1',np.nan)))
ws=pd.DataFrame(rows)
print("\n"+"="*70); print("EXPERIMENT 3 - WINDOW SIZE (group-aware validation)"); print("="*70)
print(ws.round(3).to_string(index=False))
if len(ws)>1:
    best=ws.loc[ws.f1.idxmax()]
    print(f"\nbest window by F1: {int(best.window)} beats ({best.f1:.3f})")
    print("Compare the spread against seed noise (~0.03 measured previously) before")
    print("treating any difference as real.")


window 60: 12026 windows, exp dim 10

w60_group  S02 S03 S04 S05 S06 S07 S08 S09 S10 S11 S13 S14 S15 S16 S17 
   F1=0.595   calibrated F1=0.597  [saved]

window 180: 11666 windows, exp dim 9

w180_group  S02 S03 S04 S05 S06 S07 S08 S09 S10 S11 S13 S14 S15 S16 S17 
   F1=0.522   calibrated F1=0.558  [saved]

EXPERIMENT 3 - WINDOW SIZE (group-aware validation)
 window  n_windows    f1  kappa  ma_mae  f1_cal
     60      12026 0.595  0.828   0.498   0.597
    120      11846 0.552  0.828   0.518   0.552
    180      11666 0.522  0.787   0.595   0.558

best window by F1: 60 beats (0.595)
Compare the spread against seed noise (~0.03 measured previously) before
treating any difference as real.


## 8. Reporting

**Experiment 1.** Report the group-aware number as the trustworthy result, with the caveat
that it also reflects two fewer training subjects. If it is lower than 0.543, that is the
honest figure — the previous number was produced with an uncontrolled validation set.

**Experiment 2.** Report the calibration gain only alongside the threshold stability. If
the fitted thresholds vary widely across folds, the gain is unlikely to transfer to a new
subject and should be described as such. A gain smaller than the ~0.03 seed noise measured
previously is not distinguishable from chance.

**Experiment 3.** Window comparison is confounded by differing overlap and window counts.
Report per-subject F1 alongside pooled figures, and treat differences smaller than seed
noise as inconclusive.

**What none of these change.** Two latent-deviation architectures both showed deviation
magnitude largest for relaxed windows and flat across stress levels. Improvements in
accuracy from validation, calibration or window size do not affect that mechanism finding,
and it should continue to be reported alongside any performance figure.
